# Model B — Pretrained DistilBERT on the Full Yelp Reviews Dataset

**Task:** 3-class sentiment analysis  
**Checkpoint:** `distilbert-base-uncased` (pretrained Hugging Face model, fine-tuned for classification).

This is a single end-to-end Google Colab notebook. It includes environment setup, full-data cache loading/rebuilding, exact split verification, tokenizer compatibility, model initialization, fine-tuning, validation, final test evaluation, error analysis, latency/parameter measurement, saving artifacts, and an inference module for Streamlit.

## Shared Yelp dataset and split

- Source: [Yelp Dataset on Kaggle](https://www.kaggle.com/datasets/yelp-dataset/yelp-dataset)
- File: `yelp_academic_dataset_review.json`
- Raw records counted by the supplied Model A notebook: **6,990,280**
- Valid unique reviews after the same cleaning/deduplication: **6,973,573**
- Labels: 1–2 stars → Negative (0), 3 stars → Neutral (1), 4–5 stars → Positive (2)
- Train: **6,913,573** | Validation: **20,000** | Test: **40,000**
- Seed: **42**

The notebook first reuses `yelp_model_a/tokenized` from Google Drive (or its own previously rebuilt shared cache) and verifies the exact split SHA-256 fingerprints recorded by Model A. If neither cache is available, it rebuilds the cache using the same preprocessing and split code from Model A, then checks the same fingerprints before training.

> The previous 150K balanced CSV files are NOT used in this full-data run.

## Model B vs Model A

- Model A: manually implemented encoder-only Transformer trained from random initialization.
- Model B: pretrained DistilBERT encoder + newly initialized 3-class classification head, fine-tuned with Hugging Face.
- Both models use the same review records, label IDs, split membership, and held-out test items.
- Model B is not expected to reproduce Model A's architecture; it follows the pretrained-model requirement.

## Runtime

Use **Google Colab A100 GPU + High-RAM**. The notebook selects a conservative batch size based on GPU memory. It uses the full 6.9M-example training split for one complete epoch by default and supports resumable checkpoints. Full-data fine-tuning may take several hours.

Run cells in order. Never use the held-out test set for tuning.

## Important experiment note

Model B fine-tunes DistilBERT for **one complete epoch** over all 6,913,573 training reviews by default. Model A used three epochs. The project brief requires the same dataset split and held-out test examples, not identical optimizers or epoch counts; the different training budgets are recorded and must be disclosed in the report.

The latency benchmark is aligned with Model A: the same held-out examples, 64-token padding buckets, batch sizes 1 and 32, 20 warm-up calls, 200 measured calls, and synchronized GPU timing. The exported inference helper separately reports end-to-end latency (including tokenization) and model-only latency.


## 1. Install dependencies

PyTorch is intentionally not reinstalled, to preserve the CUDA-enabled build provided by Colab.

In [ ]:
%pip -q install "transformers==4.48.3" "accelerate==1.2.1" "kagglehub>=0.3.6" orjson scikit-learn matplotlib seaborn tqdm safetensors psutil

## 2. Mount Google Drive and inspect the runtime

In [ ]:
import os, gc, json, math, time, random, shutil, hashlib, subprocess, sys
from pathlib import Path
from dataclasses import dataclass
from typing import Any

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
import psutil

from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    precision_recall_fscore_support,
)
from torch.utils.data import DataLoader, Sampler
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from google.colab import drive

drive.mount("/content/drive")

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required. Select Runtime > Change runtime type > GPU.")

GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
GPU_BF16 = torch.cuda.is_bf16_supported()
RAM_GB = psutil.virtual_memory().total / (1024 ** 3)

print(f"GPU: {GPU_NAME} | VRAM: {GPU_VRAM_GB:.1f} GB | BF16: {GPU_BF16}")
print(f"System RAM: {RAM_GB:.1f} GB | PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")

if "A100" not in GPU_NAME.upper():
    print("WARNING: A100 + High-RAM is recommended. A smaller GPU may run with smaller batches, but full-data training will take longer.")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

DRIVE_ROOT = Path("/content/drive/MyDrive")
MODEL_A_CACHE_DRIVE = DRIVE_ROOT / "yelp_model_a" / "tokenized"
MODEL_B_ROOT = DRIVE_ROOT / "yelp_model_b"
MODEL_B_SHARED_CACHE_DRIVE = MODEL_B_ROOT / "shared_tokenized_cache"
MODEL_B_CACHE = Path("/content/work/yelp_model_a_tokenized")
RAW_DIR = Path("/content/yelp_raw")
CHECKPOINT_DIR = MODEL_B_ROOT / "checkpoints"
ARTIFACTS_DIR = MODEL_B_ROOT / "artifacts"
MODEL_EXPORT_DIR = MODEL_B_ROOT / "model"

for folder in [MODEL_B_ROOT, CHECKPOINT_DIR, ARTIFACTS_DIR, MODEL_EXPORT_DIR, RAW_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Model A shared cache:", MODEL_A_CACHE_DRIVE)
print("Model B output folder:", MODEL_B_ROOT)

## 3. Reproducibility settings and expected Model A split fingerprints

In [ ]:
SEED = 42
LABEL2ID = {"negative": 0, "neutral": 1, "positive": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
CLASS_NAMES = ["negative", "neutral", "positive"]

# Exact counts and SHA-256 fingerprints read from the supplied Model A notebook output.
EXPECTED_COUNTS = {
    "raw": 6_990_280,
    "dropped_invalid": 0,
    "dropped_duplicate": 16_707,
    "deduped": 6_973_573,
}
EXPECTED_SPLITS = {
    "train": {
        "size": 6_913_573,
        "class_counts": {"negative": 1_595_427, "neutral": 684_818, "positive": 4_633_328},
        "sha256": "8245a1d054ee7b5adf5d8b1cd4c78e8849269f89555dbdae9ed50909519aa04c",
    },
    "val": {
        "size": 20_000,
        "class_counts": {"negative": 4_615, "neutral": 1_981, "positive": 13_404},
        "sha256": "54a58e3f134c464edf3fc2f56871d74c3979477d6839d2d0b41f4ff805ea578a",
    },
    "test": {
        "size": 40_000,
        "class_counts": {"negative": 9_231, "neutral": 3_962, "positive": 26_807},
        "sha256": "7a7188dfbd6112b78b68da5d640b819fe3b98b987acb1f83d5215ca256353241",
    },
}

MODEL_CHECKPOINT = "distilbert-base-uncased"
TOKENIZER_FOR_SHARED_CACHE = "bert-base-uncased"  # tokenizer used to create Model A's cache
MAX_SEQ_LEN = 256
HEAD_TOKENS = 192
TAIL_TOKENS = 64

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Labels:", LABEL2ID)
print("Model B checkpoint:", MODEL_CHECKPOINT)
print("Shared token policy:", f"{HEAD_TOKENS} head + {TAIL_TOKENS} tail = {MAX_SEQ_LEN}")

## 4. Define the exact Model A data builder

The next cell includes the dataset-building helper from the supplied Model A notebook. It is only used if Model A's tokenized cache is not present. It keeps the same star-to-label mapping, normalized-text deduplication, tokenizer, head/tail truncation, and stratified split method.

In [ ]:
# === AUTO-GENERATED FROM src/data.py ===
"""Streaming JSONL preprocessing, CSR uint16 memmap dataset, and bucketed collation."""

from array import array
import hashlib
import json
import math
from pathlib import Path
from typing import Any

import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit
import torch
from torch.utils.data import Dataset

try:
    import orjson

    def _json_loads(raw_line: bytes) -> Any:
        return orjson.loads(raw_line)

except ImportError:

    def _json_loads(raw_line: bytes) -> Any:
        return json.loads(raw_line)



def map_stars_to_label(stars: Any) -> int | None:
    """Map Yelp star rating (1..5) to 3-class ID: {1,2}->0, {3}->1, {4,5}->2."""
    if stars is None or isinstance(stars, bool):
        return None
    try:
        val = float(stars)
    except (TypeError, ValueError):
        return None
    if math.isnan(val) or math.isinf(val):
        return None

    rounded = int(round(val))
    if rounded in (1, 2):
        return 0
    if rounded == 3:
        return 1
    if rounded in (4, 5):
        return 2
    return None


def normalize_for_dedup(text: str) -> str:
    """Lowercase and collapse whitespace for exact-duplicate detection."""
    return " ".join(text.strip().lower().split())


def head_tail_truncate(
    token_ids: list[int],
    max_len: int = 256,
    head_tokens: int = 192,
    tail_tokens: int = 64,
) -> list[int]:
    """Keep leading head_tokens and trailing tail_tokens (preserving [CLS] and [SEP])."""
    if len(token_ids) <= max_len:
        return list(token_ids)
    head = token_ids[:head_tokens] if head_tokens > 0 else []
    # Guard tail_tokens == 0 because Python slices [-0:] as [0:].
    tail = token_ids[-tail_tokens:] if tail_tokens > 0 else []
    return (head + tail)[:max_len]


def _count_classes(labels_subset: np.ndarray) -> dict[str, int]:
    return {
        ID2LABEL[cid]: int(np.sum(labels_subset == cid))
        for cid in sorted(ID2LABEL.keys())
    }


def build_memmap_dataset(
    json_path: str | Path,
    out_dir: str | Path,
    tokenizer: Any,
    model_cfg: Any,
    train_cfg: Any,
) -> dict[str, Any]:
    """Stream Yelp JSONL into deduplicated CSR uint16 memmap files and stratified splits."""
    assert tokenizer.vocab_size <= 65535, (
        f"Tokenizer vocab_size ({tokenizer.vocab_size}) exceeds uint16 max (65535)"
    )

    src_path = Path(json_path)
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    tokens_bin_path = out_path / "tokens.bin"
    chunk_size = 16_384

    raw_count = 0
    dropped_invalid = 0
    dropped_duplicate = 0
    kept_count = 0
    total_tokens = 0

    seen_hashes: set[bytes] = set()
    src_offsets_buf = array("q")  # byte offset of each kept record in the source JSONL
    offsets_buf = array("q")
    lengths_buf = array("H")
    labels_buf = array("B")

    chunk_texts: list[str] = []
    chunk_labels: list[int] = []

    def _flush_chunk(tokens_file: Any) -> None:
        nonlocal kept_count, total_tokens
        if not chunk_texts:
            return
        encoded = tokenizer(
            chunk_texts,
            add_special_tokens=True,
            truncation=False,
            padding=False,
        )
        for ids, lbl in zip(encoded["input_ids"], chunk_labels):
            trunc = head_tail_truncate(
                ids,
                max_len=model_cfg.max_seq_len,
                head_tokens=model_cfg.head_tokens,
                tail_tokens=model_cfg.tail_tokens,
            )
            seq_len = len(trunc)
            offsets_buf.append(total_tokens)
            lengths_buf.append(seq_len)
            labels_buf.append(lbl)
            if seq_len > 0:
                tokens_file.write(np.asarray(trunc, dtype=np.uint16).tobytes())
            total_tokens += seq_len
            kept_count += 1
        chunk_texts.clear()
        chunk_labels.clear()

    with src_path.open("rb") as f_in, tokens_bin_path.open("wb") as f_tok:
        line_start = 0
        for raw_line in f_in:
            line_pos = line_start
            line_start += len(raw_line)
            stripped = raw_line.strip()
            if not stripped:
                continue
            raw_count += 1
            try:
                record = _json_loads(stripped)
            except (ValueError, TypeError, UnicodeDecodeError):
                dropped_invalid += 1
                continue

            if not isinstance(record, dict):
                dropped_invalid += 1
                continue

            label = map_stars_to_label(record.get("stars"))
            text = record.get("text")
            if label is None or not isinstance(text, str):
                dropped_invalid += 1
                continue

            norm_text = normalize_for_dedup(text)
            if not norm_text:
                dropped_invalid += 1
                continue

            # 64-bit keyed-free blake2b digest keeps the ~7M-entry dedup set compact in RAM.
            digest = hashlib.blake2b(norm_text.encode("utf-8"), digest_size=8).digest()
            if digest in seen_hashes:
                dropped_duplicate += 1
                continue
            seen_hashes.add(digest)

            src_offsets_buf.append(line_pos)
            chunk_texts.append(text)
            chunk_labels.append(label)
            if len(chunk_texts) >= chunk_size:
                _flush_chunk(f_tok)

        _flush_chunk(f_tok)

    if kept_count == 0:
        raise ValueError(f"No valid records found in {src_path}")
    if train_cfg.val_size + train_cfg.test_size >= kept_count:
        raise ValueError(
            f"val_size ({train_cfg.val_size}) + test_size ({train_cfg.test_size}) "
            f"must be strictly less than kept_count ({kept_count})"
        )

    offsets_arr = np.asarray(offsets_buf, dtype=np.int64)
    lengths_arr = np.asarray(lengths_buf, dtype=np.uint16)
    labels_arr = np.asarray(labels_buf, dtype=np.uint8)

    np.save(out_path / "offsets.npy", offsets_arr)
    np.save(out_path / "lengths.npy", lengths_arr)
    np.save(out_path / "labels.npy", labels_arr)

    all_indices = np.arange(kept_count, dtype=np.int32)
    sss_test = StratifiedShuffleSplit(
        n_splits=1,
        test_size=train_cfg.test_size,
        random_state=train_cfg.seed,
    )
    trainval_pos, test_pos = next(sss_test.split(all_indices, labels_arr))
    trainval_idx = all_indices[trainval_pos].astype(np.int32)
    test_idx = all_indices[test_pos].astype(np.int32)

    sss_val = StratifiedShuffleSplit(
        n_splits=1,
        test_size=train_cfg.val_size,
        random_state=train_cfg.seed,
    )
    train_rel, val_rel = next(sss_val.split(trainval_idx, labels_arr[trainval_idx]))
    train_idx = trainval_idx[train_rel].astype(np.int32)
    val_idx = trainval_idx[val_rel].astype(np.int32)

    np.save(out_path / "train_idx.npy", train_idx)
    np.save(out_path / "val_idx.npy", val_idx)
    np.save(out_path / "test_idx.npy", test_idx)

    # Export original (untruncated, cased) test review texts in test_idx order so error
    # analysis shows real text instead of lowercased, head+tail-truncated decoded tokens.
    src_offsets = np.asarray(src_offsets_buf, dtype=np.int64)
    test_texts: list[str] = [""] * len(test_idx)
    read_order = np.argsort(src_offsets[test_idx], kind="mergesort")  # sequential disk reads
    with src_path.open("rb") as f_src:
        for j in read_order:
            f_src.seek(int(src_offsets[test_idx[j]]))
            test_texts[int(j)] = _json_loads(f_src.readline().strip())["text"]
    with (out_path / "test_texts.jsonl").open("w", encoding="utf-8") as f_txt:
        for t in test_texts:
            f_txt.write(json.dumps({"text": t}, ensure_ascii=False) + "\n")

    manifest: dict[str, Any] = {
        "dataset_slug": train_cfg.dataset_slug,
        "dataset_file": train_cfg.dataset_file,
        "dataset_url": train_cfg.dataset_url,
        "dataset_license": train_cfg.dataset_license,
        "citation": (
            "Yelp Open Dataset (https://www.yelp.com/dataset), "
            f"mirror: {train_cfg.dataset_url}"
        ),
        "tokenizer_name": train_cfg.tokenizer_name,
        "seed": train_cfg.seed,
        "max_seq_len": model_cfg.max_seq_len,
        "head_tokens": model_cfg.head_tokens,
        "tail_tokens": model_cfg.tail_tokens,
        "counts": {
            "raw": raw_count,
            "dropped_invalid": dropped_invalid,
            "dropped_duplicate": dropped_duplicate,
            "deduped": kept_count,
            "total_tokens": total_tokens,
        },
        "splits": {
            "train": {
                "size": int(len(train_idx)),
                "class_counts": _count_classes(labels_arr[train_idx]),
                "sha256": hashlib.sha256(train_idx.tobytes()).hexdigest(),
            },
            "val": {
                "size": int(len(val_idx)),
                "class_counts": _count_classes(labels_arr[val_idx]),
                "sha256": hashlib.sha256(val_idx.tobytes()).hexdigest(),
            },
            "test": {
                "size": int(len(test_idx)),
                "class_counts": _count_classes(labels_arr[test_idx]),
                "sha256": hashlib.sha256(test_idx.tobytes()).hexdigest(),
            },
        },
    }

    with (out_path / "split_manifest.json").open("w", encoding="utf-8") as f_manifest:
        json.dump(manifest, f_manifest, indent=2)

    return manifest


class YelpMemmapDataset(Dataset):
    """CSR memmap dataset that opens read-only file mappings lazily per worker."""

    def __init__(self, data_dir: str | Path, indices: np.ndarray) -> None:
        base = Path(data_dir)
        self.tokens_path = str(base / "tokens.bin")
        self.offsets_path = str(base / "offsets.npy")
        self.lengths_path = str(base / "lengths.npy")
        self.labels_path = str(base / "labels.npy")
        self.indices = np.asarray(indices, dtype=np.int64)

        # Defer mmap creation until __getitem__ so DataLoader fork/spawn workers
        # never share open mmap descriptors across processes.
        self._tokens: np.memmap | None = None
        self._offsets: np.ndarray | None = None
        self._lengths: np.ndarray | None = None
        self._labels: np.ndarray | None = None

    def _ensure_open(self) -> None:
        if self._tokens is None:
            self._tokens = np.memmap(self.tokens_path, dtype=np.uint16, mode="r")
            self._offsets = np.load(self.offsets_path, mmap_mode="r")
            self._lengths = np.load(self.lengths_path, mmap_mode="r")
            self._labels = np.load(self.labels_path, mmap_mode="r")

    def __len__(self) -> int:
        return int(len(self.indices))

    def __getitem__(self, item: int) -> tuple[np.ndarray, int]:
        self._ensure_open()
        assert self._tokens is not None
        assert self._offsets is not None
        assert self._lengths is not None
        assert self._labels is not None

        idx = int(self.indices[item])
        s = int(self._offsets[idx])
        e = s + int(self._lengths[idx])
        # Copy slice into a writable int64 array to avoid read-only memmap warnings.
        tokens = np.array(self._tokens[s:e], dtype=np.int64)
        return tokens, int(self._labels[idx])


class BucketCollateFn:
    """Pad variable-length sequences to multiples of bucket_multiple for torch.compile."""

    def __init__(
        self,
        pad_token_id: int = 0,
        bucket_multiple: int = 64,
        max_seq_len: int = 256,
    ) -> None:
        if bucket_multiple <= 0 or max_seq_len <= 0:
            raise ValueError("bucket_multiple and max_seq_len must be positive")
        self.pad_token_id = pad_token_id
        self.bucket_multiple = bucket_multiple
        self.max_seq_len = max_seq_len

    def __call__(
        self,
        batch: list[tuple[np.ndarray, int]],
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        if not batch:
            return (
                torch.empty((0, 0), dtype=torch.long),
                torch.empty((0, 0), dtype=torch.bool),
                torch.empty((0,), dtype=torch.long),
            )

        seqs, labels = zip(*batch)
        max_len_in_batch = max(len(s) for s in seqs)

        # Snap padded length to a fixed set of multiples of bucket_multiple (e.g. 64, 128, 192, 256)
        # so torch.compile sees at most max_seq_len // bucket_multiple distinct input shapes.
        bucketed = (
            (max_len_in_batch + self.bucket_multiple - 1) // self.bucket_multiple
        ) * self.bucket_multiple
        padded_len = min(self.max_seq_len, max(self.bucket_multiple, bucketed))

        bsz = len(seqs)
        input_ids = torch.full((bsz, padded_len), self.pad_token_id, dtype=torch.long)
        attention_mask = torch.zeros((bsz, padded_len), dtype=torch.bool)
        labels_tensor = torch.tensor(labels, dtype=torch.long)

        for i, seq in enumerate(seqs):
            seq_len = min(len(seq), padded_len)
            if seq_len > 0:
                input_ids[i, :seq_len] = torch.as_tensor(seq[:seq_len], dtype=torch.long)
                attention_mask[i, :seq_len] = True

        return input_ids, attention_mask, labels_tensor


## 5. Reuse or rebuild the full Yelp cache

In [ ]:
@dataclass
class SharedDataModelConfig:
    max_seq_len: int = MAX_SEQ_LEN
    head_tokens: int = HEAD_TOKENS
    tail_tokens: int = TAIL_TOKENS

@dataclass
class SharedDataTrainConfig:
    seed: int = SEED
    val_size: int = 20_000
    test_size: int = 40_000
    tokenizer_name: str = TOKENIZER_FOR_SHARED_CACHE
    dataset_slug: str = "yelp-dataset/yelp-dataset"
    dataset_file: str = "yelp_academic_dataset_review.json"
    dataset_url: str = "https://www.kaggle.com/datasets/yelp-dataset/yelp-dataset"
    dataset_license: str = "Yelp Dataset Agreement (non-commercial academic/educational use)"

data_model_cfg = SharedDataModelConfig()
data_train_cfg = SharedDataTrainConfig()

REQUIRED_CACHE_FILES = (
    "tokens.bin", "offsets.npy", "lengths.npy", "labels.npy",
    "train_idx.npy", "val_idx.npy", "test_idx.npy", "split_manifest.json",
)
OPTIONAL_CACHE_FILES = ("test_texts.jsonl",)

def cache_is_complete(cache_dir: Path) -> bool:
    if not all((cache_dir / name).exists() for name in REQUIRED_CACHE_FILES):
        return False
    try:
        manifest = json.loads((cache_dir / "split_manifest.json").read_text(encoding="utf-8"))
        counts = manifest["counts"]
        if (cache_dir / "tokens.bin").stat().st_size != int(counts["total_tokens"]) * 2:
            return False
        for name in ("offsets.npy", "lengths.npy", "labels.npy"):
            if len(np.load(cache_dir / name, mmap_mode="r")) != int(counts["deduped"]):
                return False
        for split_name in ("train", "val", "test"):
            arr = np.load(cache_dir / f"{split_name}_idx.npy", mmap_mode="r")
            if len(arr) != int(manifest["splits"][split_name]["size"]):
                return False
    except Exception as exc:
        print("Cache integrity check failed:", repr(exc))
        return False
    return True

def copy_shared_cache(src_dir: Path, dst_dir: Path):
    dst_dir.mkdir(parents=True, exist_ok=True)
    for name in OPTIONAL_CACHE_FILES:
        if (src_dir / name).exists():
            shutil.copy2(src_dir / name, dst_dir / name)
    for name in REQUIRED_CACHE_FILES:
        shutil.copy2(src_dir / name, dst_dir / name)

def configure_kaggle_auth():
    # Prefer Colab Secrets. Credentials are stored only in this temporary runtime.
    username = key = None
    try:
        from google.colab import userdata
        username = userdata.get("KAGGLE_USERNAME")
        key = userdata.get("KAGGLE_KEY")
    except Exception:
        pass

    local_json = Path.home() / ".kaggle" / "kaggle.json"
    if (not username or not key) and local_json.exists():
        creds = json.loads(local_json.read_text(encoding="utf-8"))
        username, key = creds.get("username"), creds.get("key")

    if not username or not key:
        from google.colab import files
        print("Upload kaggle.json to this temporary Colab session. It will NOT be copied to Drive.")
        uploaded = files.upload()
        if "kaggle.json" not in uploaded:
            raise RuntimeError("Please upload a file named kaggle.json.")
        creds = json.loads(uploaded["kaggle.json"].decode("utf-8"))
        username, key = creds.get("username"), creds.get("key")

    if not username or not key:
        raise RuntimeError("kaggle.json must contain username and key.")

    os.environ["KAGGLE_USERNAME"] = str(username)
    os.environ["KAGGLE_KEY"] = str(key)
    local_json.parent.mkdir(parents=True, exist_ok=True)
    local_json.write_text(json.dumps({"username": username, "key": key}), encoding="utf-8")
    os.chmod(local_json, 0o600)
    print("Kaggle authentication configured in the temporary runtime.")

def ensure_full_dataset_cache():
    if cache_is_complete(MODEL_B_CACHE):
        print("Using the complete local tokenized cache.")
        return

    if MODEL_B_CACHE.exists():
        shutil.rmtree(MODEL_B_CACHE)
    MODEL_B_CACHE.mkdir(parents=True, exist_ok=True)

    source_cache = None
    if cache_is_complete(MODEL_A_CACHE_DRIVE):
        source_cache = MODEL_A_CACHE_DRIVE
        print("Found Model A's full tokenized cache on Drive.")
    elif cache_is_complete(MODEL_B_SHARED_CACHE_DRIVE):
        source_cache = MODEL_B_SHARED_CACHE_DRIVE
        print("Found the previously rebuilt shared cache in Model B's Drive folder.")

    if source_cache is not None:
        free_gb = shutil.disk_usage("/content").free / (1024 ** 3)
        if free_gb < 5:
            raise RuntimeError(f"Not enough local Colab disk space ({free_gb:.1f} GB free).")
        print("Copying the shared tokenized cache to local Colab storage...")
        copy_shared_cache(source_cache, MODEL_B_CACHE)
        print("Shared cache copied.")
        return

    print("A verified shared cache is not available. Rebuilding it using Model A's exact data builder.")
    if RAM_GB and RAM_GB < 40:
        raise RuntimeError(
            f"Only {RAM_GB:.1f} GB system RAM detected. Select Colab High-RAM before rebuilding "
            "the full 7M-review cache, or ask the Model A teammate to share yelp_model_a/tokenized."
        )
    raw_json_path = RAW_DIR / data_train_cfg.dataset_file
    if not raw_json_path.exists():
        configure_kaggle_auth()
        import kagglehub
        try:
            downloaded = kagglehub.dataset_download(
                data_train_cfg.dataset_slug,
                path=data_train_cfg.dataset_file,
            )
        except Exception as exc:
            raise RuntimeError(
                "Kaggle download failed. Open the Yelp dataset page, accept its terms if prompted, "
                "then rerun this cell. Original error: " + repr(exc)
            ) from exc
        downloaded_path = Path(downloaded)
        raw_json_path = downloaded_path / data_train_cfg.dataset_file if downloaded_path.is_dir() else downloaded_path

    if not raw_json_path.exists():
        raise FileNotFoundError(f"Could not find {data_train_cfg.dataset_file}: {raw_json_path}")
    raw_size = raw_json_path.stat().st_size
    if raw_size < 5_000_000_000:
        raise RuntimeError(f"Expected full Yelp JSON (>5 GB), found {raw_size / 1e9:.2f} GB.")

    shared_tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_FOR_SHARED_CACHE, use_fast=True)
    manifest = build_memmap_dataset(
        json_path=raw_json_path,
        out_dir=MODEL_B_CACHE,
        tokenizer=shared_tokenizer,
        model_cfg=data_model_cfg,
        train_cfg=data_train_cfg,
    )
    print("Rebuilt full dataset cache:", manifest["counts"])
    print("Backing up the verified shared cache to Model B's Google Drive folder...")
    copy_shared_cache(MODEL_B_CACHE, MODEL_B_SHARED_CACHE_DRIVE)
    print("Shared cache backup complete:", MODEL_B_SHARED_CACHE_DRIVE)

ensure_full_dataset_cache()
assert cache_is_complete(MODEL_B_CACHE), "The tokenized dataset cache is incomplete."
print("Full cache ready:", MODEL_B_CACHE)

## 6. Verify exact split membership and class distributions

In [ ]:
manifest = json.loads((MODEL_B_CACHE / "split_manifest.json").read_text(encoding="utf-8"))

assert manifest["dataset_slug"] == "yelp-dataset/yelp-dataset"
assert manifest["dataset_file"] == "yelp_academic_dataset_review.json"
assert int(manifest["seed"]) == SEED
assert manifest["tokenizer_name"] == TOKENIZER_FOR_SHARED_CACHE
assert int(manifest["max_seq_len"]) == MAX_SEQ_LEN
assert int(manifest["head_tokens"]) == HEAD_TOKENS
assert int(manifest["tail_tokens"]) == TAIL_TOKENS

for key, expected in EXPECTED_COUNTS.items():
    actual = int(manifest["counts"][key])
    assert actual == expected, f"{key}: expected {expected:,}, found {actual:,}"

labels_memmap = np.load(MODEL_B_CACHE / "labels.npy", mmap_mode="r")
deduped_count = int(manifest["counts"]["deduped"])
assert len(labels_memmap) == deduped_count
assert set(np.unique(labels_memmap).tolist()).issubset({0, 1, 2}), "Unexpected label IDs found."

split_arrays = {}
for split_name, expected in EXPECTED_SPLITS.items():
    indices = np.load(MODEL_B_CACHE / f"{split_name}_idx.npy")
    digest = hashlib.sha256(indices.tobytes()).hexdigest()

    assert indices.dtype == np.int32, f"{split_name} indices must match Model A's int32 format."
    assert len(indices) == expected["size"], f"{split_name} size mismatch."
    assert int(indices.min()) >= 0 and int(indices.max()) < deduped_count, (
        f"{split_name} contains out-of-range record indices."
    )
    assert digest == expected["sha256"], (
        f"{split_name} SHA-256 differs from Model A. Stop here; do not train on a mismatched split."
    )

    actual_counts = {
        ID2LABEL[i]: int(np.sum(labels_memmap[indices] == i))
        for i in range(3)
    }
    assert actual_counts == expected["class_counts"], (
        f"{split_name} class counts differ: {actual_counts}"
    )
    split_arrays[split_name] = indices
    print(f"{split_name.upper():>5}: {len(indices):,} | SHA-256 PASS | {actual_counts}")

train_idx = split_arrays["train"]
val_idx = split_arrays["val"]
test_idx = split_arrays["test"]

assert np.intersect1d(train_idx, val_idx).size == 0
assert np.intersect1d(train_idx, test_idx).size == 0
assert np.intersect1d(val_idx, test_idx).size == 0

all_indices = np.concatenate([train_idx, val_idx, test_idx])
assert len(all_indices) == deduped_count
assert len(np.unique(all_indices)) == deduped_count, "The three splits do not cover every unique review exactly once."

print("\nPASS: exact Model A split membership, label distributions, no overlap, and full coverage verified.")
print("No 150K balanced sample is used in this notebook.")

## 7. Verify DistilBERT tokenizer compatibility

Model A created its cached token IDs with `bert-base-uncased`. DistilBERT base uncased is expected to use the same WordPiece vocabulary and special token IDs. We explicitly compare the full vocabularies before using those cached IDs.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, use_fast=True)
model_a_tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_FOR_SHARED_CACHE, use_fast=True)

assert tokenizer.vocab_size == 30_522, f"Unexpected vocabulary size: {tokenizer.vocab_size}"
assert tokenizer.get_vocab() == model_a_tokenizer.get_vocab(), (
    "Tokenizer vocabularies differ. Do not train using Model A token IDs."
)
assert tokenizer.pad_token_id == 0
assert tokenizer.cls_token_id == 101
assert tokenizer.sep_token_id == 102

# Confirm that both tokenizers produce the exact same IDs for representative text.
tokenizer_probe_texts = [
    "The food was excellent, and the staff were friendly!",
    "I wouldn't recommend this place. The service was VERY slow.",
    "Café crème, punctuation... and mixed-case TEXT.",
    "   Extra spaces\tand a newline\nshould normalize consistently.   ",
]
for probe in tokenizer_probe_texts:
    ids_b = tokenizer(probe, add_special_tokens=True, truncation=False)["input_ids"]
    ids_a = model_a_tokenizer(probe, add_special_tokens=True, truncation=False)["input_ids"]
    assert ids_b == ids_a, f"Tokenizer output mismatch for probe text: {probe!r}"

print("PASS: Model A and DistilBERT vocabularies, special IDs, and tokenization outputs are identical.")
print("Vocabulary size:", tokenizer.vocab_size)
print("Special IDs:", {"PAD": tokenizer.pad_token_id, "CLS": tokenizer.cls_token_id, "SEP": tokenizer.sep_token_id})

## 8. Create memory-mapped datasets and dynamic padding

In [ ]:
class DistilBertCollator:
    def __init__(self, pad_token_id: int, max_length: int = 256, pad_multiple: int = 8):
        self.pad_token_id = int(pad_token_id)
        self.max_length = int(max_length)
        self.pad_multiple = int(pad_multiple)

    def __call__(self, batch):
        sequences, labels = zip(*batch)
        longest = min(self.max_length, max(len(x) for x in sequences))
        padded_len = min(
            self.max_length,
            max(self.pad_multiple, math.ceil(longest / self.pad_multiple) * self.pad_multiple),
        )
        input_ids = torch.full((len(batch), padded_len), self.pad_token_id, dtype=torch.long)
        attention_mask = torch.zeros((len(batch), padded_len), dtype=torch.long)
        for row, seq in enumerate(sequences):
            seq = seq[:padded_len]
            n = len(seq)
            input_ids[row, :n] = torch.as_tensor(seq, dtype=torch.long)
            attention_mask[row, :n] = 1
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": torch.tensor(labels, dtype=torch.long),
        }

train_dataset = YelpMemmapDataset(MODEL_B_CACHE, train_idx)
val_dataset = YelpMemmapDataset(MODEL_B_CACHE, val_idx)
test_dataset = YelpMemmapDataset(MODEL_B_CACHE, test_idx)
collator = DistilBertCollator(tokenizer.pad_token_id, MAX_SEQ_LEN, 8)

print(f"Train: {len(train_dataset):,} | Validation: {len(val_dataset):,} | Test: {len(test_dataset):,}")

## 9. Initialize the pretrained DistilBERT sequence classifier

In [ ]:
# Conservative automatic batch size. A100 40GB selects 32 examples per micro-batch.
if GPU_VRAM_GB >= 70:
    PER_DEVICE_BATCH_SIZE = 64
elif GPU_VRAM_GB >= 35:
    PER_DEVICE_BATCH_SIZE = 32
elif GPU_VRAM_GB >= 20:
    PER_DEVICE_BATCH_SIZE = 16
elif GPU_VRAM_GB >= 12:
    PER_DEVICE_BATCH_SIZE = 8
else:
    raise RuntimeError("At least 12 GB GPU memory is required; A100 40GB+ is recommended.")

GRAD_ACCUM_STEPS = 2
EFFECTIVE_BATCH_SIZE = PER_DEVICE_BATCH_SIZE * GRAD_ACCUM_STEPS
EVAL_BATCH_SIZE = min(64, max(8, PER_DEVICE_BATCH_SIZE * 2))
NUM_EPOCHS = 1  # one complete pass through all 6,913,573 training examples
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
MAX_GRAD_NORM = 1.0
VALIDATE_EVERY_STEPS = 10_000
CHECKPOINT_EVERY_STEPS = 5_000
NUM_WORKERS = min(4, os.cpu_count() or 2)
RESET_TRAINING = False  # set True only when intentionally starting a fresh run

device = torch.device("cuda")
USE_BF16 = bool(GPU_BF16)
USE_FP16 = not USE_BF16
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=3,
    id2label={i: label for i, label in ID2LABEL.items()},
    label2id=LABEL2ID,
    problem_type="single_label_classification",
)
model.config.id2label = {i: label for i, label in ID2LABEL.items()}
model.config.label2id = LABEL2ID
model.config.problem_type = "single_label_classification"
model.to(device)

if GPU_VRAM_GB < 30:
    model.gradient_checkpointing_enable()
    print("Gradient checkpointing enabled for the available GPU.")

total_parameters = sum(p.numel() for p in model.parameters())
trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Checkpoint:", MODEL_CHECKPOINT)
print(f"Total parameters: {total_parameters:,}")
print(f"Trainable parameters: {trainable_parameters:,}")
print(f"Micro-batch: {PER_DEVICE_BATCH_SIZE} | Effective batch: {EFFECTIVE_BATCH_SIZE}")
print(f"Epochs: {NUM_EPOCHS} | Mixed precision: {'BF16' if USE_BF16 else 'FP16'}")

## 10. Build resumable full-data batches, optimizer, and learning-rate scheduler

In [ ]:
class EpochBatchSampler(Sampler):
    """Deterministic shuffled batches; start_batch makes epoch-level resume possible."""
    def __init__(self, n_items, batch_size, seed, epoch, start_batch=0):
        self.n_items = int(n_items)
        self.batch_size = int(batch_size)
        self.total_batches = math.ceil(self.n_items / self.batch_size)
        self.start_batch = int(start_batch)
        generator = torch.Generator(device="cpu")
        generator.manual_seed(int(seed) + int(epoch))
        self.permutation = torch.randperm(self.n_items, generator=generator).numpy()

    def __iter__(self):
        for batch_idx in range(self.start_batch, self.total_batches):
            start = batch_idx * self.batch_size
            end = min(start + self.batch_size, self.n_items)
            yield self.permutation[start:end].tolist()

    def __len__(self):
        return max(0, self.total_batches - self.start_batch)

batches_per_epoch = math.ceil(len(train_dataset) / PER_DEVICE_BATCH_SIZE)
optimizer_steps_per_epoch = math.ceil(batches_per_epoch / GRAD_ACCUM_STEPS)
total_training_steps = optimizer_steps_per_epoch * NUM_EPOCHS
warmup_steps = max(1, int(total_training_steps * WARMUP_RATIO))

decay_params, no_decay_params = [], []
for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if name.endswith(".bias") or "LayerNorm.weight" in name or "layer_norm.weight" in name:
        no_decay_params.append(param)
    else:
        decay_params.append(param)

optimizer = torch.optim.AdamW(
    [
        {"params": decay_params, "weight_decay": WEIGHT_DECAY},
        {"params": no_decay_params, "weight_decay": 0.0},
    ],
    lr=LEARNING_RATE,
    betas=(0.9, 0.999),
    eps=1e-8,
)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps,
)
try:
    scaler = torch.amp.GradScaler("cuda", enabled=USE_FP16)
except (AttributeError, TypeError):
    scaler = torch.cuda.amp.GradScaler(enabled=USE_FP16)
criterion = nn.CrossEntropyLoss()

def make_loader(dataset, batch_sampler=None, batch_size=None, shuffle=False):
    kwargs = {
        "dataset": dataset,
        "num_workers": NUM_WORKERS,
        "pin_memory": True,
        "collate_fn": collator,
    }
    if NUM_WORKERS > 0:
        kwargs["persistent_workers"] = True
        kwargs["prefetch_factor"] = 2
    if batch_sampler is not None:
        kwargs["batch_sampler"] = batch_sampler
    else:
        kwargs["batch_size"] = batch_size
        kwargs["shuffle"] = shuffle
    return DataLoader(**kwargs)

val_loader = make_loader(val_dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False)
test_loader = make_loader(test_dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False)

preview = next(iter(val_loader))
assert preview["input_ids"].shape == preview["attention_mask"].shape
assert preview["labels"].min().item() >= 0 and preview["labels"].max().item() <= 2
assert int(preview["input_ids"].max()) < tokenizer.vocab_size

# Run a tiny forward-pass preflight before starting a multi-hour training job.
preflight_ids = preview["input_ids"][:2].to(device)
preflight_mask = preview["attention_mask"][:2].to(device)
with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
    preflight_logits = model(input_ids=preflight_ids, attention_mask=preflight_mask).logits
assert tuple(preflight_logits.shape) == (2, 3), f"Unexpected model output shape: {preflight_logits.shape}"
del preflight_ids, preflight_mask, preflight_logits
torch.cuda.empty_cache()

print("Batch preview:", {k: tuple(v.shape) for k, v in preview.items()})
print("PASS: DistilBERT forward preflight returned 2 x 3 logits.")
print(f"Full training micro-batches per epoch: {batches_per_epoch:,}")
print(f"Optimizer steps per epoch: {optimizer_steps_per_epoch:,}")
print(f"Warmup steps: {warmup_steps:,}")
print("Training examples omitted: 0 (the final batch is allowed to be smaller).")

## 11. Evaluation helpers and atomic Drive checkpointing

In [ ]:
RESUME_CHECKPOINT_PATH = CHECKPOINT_DIR / "training_state.pt"
BEST_STATE_PATH = CHECKPOINT_DIR / "best_model_state.pt"
LOCAL_CHECKPOINT_TMP = Path("/content/training_state.tmp.pt")
LOCAL_BEST_TMP = Path("/content/best_model_state.tmp.pt")

TRAINING_CONFIG = {
    "model_checkpoint": MODEL_CHECKPOINT,
    "seed": SEED,
    "max_seq_len": MAX_SEQ_LEN,
    "per_device_batch_size": PER_DEVICE_BATCH_SIZE,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "num_epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "validate_every_steps": VALIDATE_EVERY_STEPS,
    "checkpoint_every_steps": CHECKPOINT_EVERY_STEPS,
    "gpu_name": GPU_NAME,
    "use_bf16": USE_BF16,
    "use_fp16": USE_FP16,
    "train_split_sha256": EXPECTED_SPLITS["train"]["sha256"],
    "val_split_sha256": EXPECTED_SPLITS["val"]["sha256"],
    "test_split_sha256": EXPECTED_SPLITS["test"]["sha256"],
}
CONFIG_FINGERPRINT = hashlib.sha256(
    json.dumps(TRAINING_CONFIG, sort_keys=True).encode("utf-8")
).hexdigest()

if RESET_TRAINING:
    for path in [RESUME_CHECKPOINT_PATH, BEST_STATE_PATH]:
        if path.exists():
            path.unlink()
    print("Old Model B checkpoints removed; a fresh run will start.")

def amp_context():
    return torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=True)

@torch.inference_mode()
def evaluate_loader(eval_model, loader):
    eval_model.eval()
    predictions, labels_all = [], []
    total_loss, total_items = 0.0, 0
    for batch in tqdm(loader, desc="Evaluating", leave=False):
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)
        with amp_context():
            logits = eval_model(input_ids=input_ids, attention_mask=attention_mask).logits
            loss = criterion(logits.float(), labels)
        n = labels.size(0)
        total_loss += float(loss.item()) * n
        total_items += n
        predictions.append(logits.argmax(dim=-1).cpu().numpy())
        labels_all.append(labels.cpu().numpy())

    y_true = np.concatenate(labels_all)
    y_pred = np.concatenate(predictions)
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1, 2], average="macro", zero_division=0
    )
    return (
        total_loss / max(total_items, 1),
        {
            "accuracy": float(accuracy_score(y_true, y_pred)),
            "macro_precision": float(p),
            "macro_recall": float(r),
            "macro_f1": float(f1),
        },
        y_true,
        y_pred,
    )

def atomic_save_to_drive(payload, drive_path: Path, local_tmp: Path):
    torch.save(payload, local_tmp)
    drive_path.parent.mkdir(parents=True, exist_ok=True)
    drive_tmp = drive_path.with_suffix(drive_path.suffix + ".tmp")
    shutil.copy2(local_tmp, drive_tmp)
    os.replace(drive_tmp, drive_path)

def save_best_model_state(eval_model, score, step):
    state_cpu = {k: v.detach().cpu() for k, v in eval_model.state_dict().items()}
    atomic_save_to_drive(
        {"model_state_dict": state_cpu, "best_val_macro_f1": float(score), "global_step": int(step)},
        BEST_STATE_PATH,
        LOCAL_BEST_TMP,
    )
    print(f"Saved best validation weights: Macro-F1={score:.5f}, step={step:,}")

def save_resume_state(epoch, next_batch_idx, global_step, best_val_macro_f1,
                      history, elapsed_train_sec, training_complete=False):
    payload = {
        "config_fingerprint": CONFIG_FINGERPRINT,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "epoch": int(epoch),
        "next_batch_idx": int(next_batch_idx),
        "global_step": int(global_step),
        "best_val_macro_f1": float(best_val_macro_f1),
        "history": history,
        "elapsed_train_sec": float(elapsed_train_sec),
        "training_complete": bool(training_complete),
        "rng_torch": torch.get_rng_state(),
        "rng_cuda": torch.cuda.get_rng_state_all(),
        "rng_numpy": np.random.get_state(),
        "rng_python": random.getstate(),
    }
    atomic_save_to_drive(payload, RESUME_CHECKPOINT_PATH, LOCAL_CHECKPOINT_TMP)

print("Resume checkpoint:", RESUME_CHECKPOINT_PATH)
print("Best weights:", BEST_STATE_PATH)
print("Config fingerprint:", CONFIG_FINGERPRINT)

## 12. Fine-tune DistilBERT on all 6,913,573 training reviews

- Full training split; no 150K sampling.
- Validation Macro-F1 selects the best checkpoint.
- A resumable checkpoint is saved periodically to Google Drive.
- If Colab disconnects, rerun the notebook with the same runtime batch settings; it restores the optimizer, scheduler, scaler, random states, and next batch.

In [ ]:
start_epoch = 0
start_batch_idx = 0
global_step = 0
best_val_macro_f1 = -1.0
history = []
elapsed_train_sec = 0.0
training_complete = False

if RESUME_CHECKPOINT_PATH.exists():
    resume_state = torch.load(RESUME_CHECKPOINT_PATH, map_location="cpu", weights_only=False)
    if resume_state.get("config_fingerprint") != CONFIG_FINGERPRINT:
        raise RuntimeError(
            "Existing checkpoint settings do not match this run. Restore the previous settings "
            "or set RESET_TRAINING=True to start a new experiment."
        )
    model.load_state_dict(resume_state["model_state_dict"])
    optimizer.load_state_dict(resume_state["optimizer_state_dict"])
    scheduler.load_state_dict(resume_state["scheduler_state_dict"])
    if resume_state.get("scaler_state_dict"):
        scaler.load_state_dict(resume_state["scaler_state_dict"])
    start_epoch = int(resume_state["epoch"])
    start_batch_idx = int(resume_state["next_batch_idx"])
    global_step = int(resume_state["global_step"])
    best_val_macro_f1 = float(resume_state["best_val_macro_f1"])
    history = list(resume_state["history"])
    elapsed_train_sec = float(resume_state["elapsed_train_sec"])
    training_complete = bool(resume_state["training_complete"])
    if not training_complete:
        torch.set_rng_state(resume_state["rng_torch"])
        torch.cuda.set_rng_state_all(resume_state["rng_cuda"])
        np.random.set_state(resume_state["rng_numpy"])
        random.setstate(resume_state["rng_python"])
    print(
        f"Restored: epoch={start_epoch}, next batch={start_batch_idx:,}, "
        f"step={global_step:,}, complete={training_complete}"
    )
else:
    print("No checkpoint found. Starting a new full-data fine-tuning run.")
    if BEST_STATE_PATH.exists():
        BEST_STATE_PATH.unlink()
        print("Removed orphaned best weights from an older run to prevent stale-model evaluation.")

if training_complete:
    print("The full training run is already marked complete; skipping training.")
else:
    for epoch in range(start_epoch, NUM_EPOCHS):
        epoch_start_batch = start_batch_idx if epoch == start_epoch else 0
        batch_sampler = EpochBatchSampler(
            n_items=len(train_dataset),
            batch_size=PER_DEVICE_BATCH_SIZE,
            seed=SEED,
            epoch=epoch,
            start_batch=epoch_start_batch,
        )
        train_loader = make_loader(train_dataset, batch_sampler=batch_sampler)
        total_batches = math.ceil(len(train_dataset) / PER_DEVICE_BATCH_SIZE)

        model.train()
        optimizer.zero_grad(set_to_none=True)
        running_loss = torch.zeros((), device=device)
        running_batches = 0
        segment_start = time.perf_counter()

        progress = tqdm(
            enumerate(train_loader, start=epoch_start_batch),
            total=total_batches - epoch_start_batch,
            desc=f"DistilBERT epoch {epoch + 1}/{NUM_EPOCHS}",
        )

        for batch_idx, batch in progress:
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)

            group_start = (batch_idx // GRAD_ACCUM_STEPS) * GRAD_ACCUM_STEPS
            group_end = min(group_start + GRAD_ACCUM_STEPS, total_batches)
            group_total_examples = sum(
                min(PER_DEVICE_BATCH_SIZE, len(train_dataset) - j * PER_DEVICE_BATCH_SIZE)
                for j in range(group_start, group_end)
            )

            with amp_context():
                logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
                loss = criterion(logits.float(), labels)
                loss_for_backward = loss * (labels.size(0) / group_total_examples)

            scaler.scale(loss_for_backward).backward()
            running_loss += loss.detach().float()
            running_batches += 1

            optimizer_boundary = (
                ((batch_idx + 1) % GRAD_ACCUM_STEPS == 0)
                or (batch_idx + 1 == total_batches)
            )

            if optimizer_boundary:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                global_step += 1

                if global_step % 100 == 0:
                    progress.set_postfix(
                        loss=f"{running_loss.item() / max(running_batches, 1):.4f}",
                        lr=f"{scheduler.get_last_lr()[0]:.2e}",
                        step=global_step,
                    )
                    running_loss = torch.zeros((), device=device)
                    running_batches = 0

                if global_step % VALIDATE_EVERY_STEPS == 0:
                    val_loss, val_metrics, _, _ = evaluate_loader(model, val_loader)
                    history.append({
                        "epoch": epoch + 1,
                        "global_step": global_step,
                        "val_loss": val_loss,
                        "val_accuracy": val_metrics["accuracy"],
                        "val_macro_precision": val_metrics["macro_precision"],
                        "val_macro_recall": val_metrics["macro_recall"],
                        "val_macro_f1": val_metrics["macro_f1"],
                    })
                    print(
                        f"[step {global_step:,}] val_loss={val_loss:.4f} | "
                        f"accuracy={val_metrics['accuracy']:.4f} | "
                        f"macro-F1={val_metrics['macro_f1']:.4f}"
                    )
                    if val_metrics["macro_f1"] > best_val_macro_f1:
                        best_val_macro_f1 = val_metrics["macro_f1"]
                        save_best_model_state(model, best_val_macro_f1, global_step)
                    model.train()

                if global_step % CHECKPOINT_EVERY_STEPS == 0:
                    elapsed_train_sec += time.perf_counter() - segment_start
                    save_resume_state(
                        epoch, batch_idx + 1, global_step, best_val_macro_f1,
                        history, elapsed_train_sec, training_complete=False,
                    )
                    segment_start = time.perf_counter()
                    print(f"Resumable checkpoint saved at step {global_step:,}.")

        elapsed_train_sec += time.perf_counter() - segment_start

        # End-of-epoch validation; test data is never used for model selection.
        val_loss, val_metrics, _, _ = evaluate_loader(model, val_loader)
        history.append({
            "epoch": epoch + 1,
            "global_step": global_step,
            "val_loss": val_loss,
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_precision": val_metrics["macro_precision"],
            "val_macro_recall": val_metrics["macro_recall"],
            "val_macro_f1": val_metrics["macro_f1"],
        })
        print(
            f"[epoch {epoch + 1} complete] val_loss={val_loss:.4f} | "
            f"accuracy={val_metrics['accuracy']:.4f} | macro-F1={val_metrics['macro_f1']:.4f}"
        )
        if val_metrics["macro_f1"] > best_val_macro_f1 or not BEST_STATE_PATH.exists():
            best_val_macro_f1 = val_metrics["macro_f1"]
            save_best_model_state(model, best_val_macro_f1, global_step)

        is_complete = (epoch + 1) >= NUM_EPOCHS
        save_resume_state(
            epoch + 1, 0, global_step, best_val_macro_f1,
            history, elapsed_train_sec, training_complete=is_complete,
        )
        start_batch_idx = 0
        print(f"Epoch checkpoint saved. Training complete={is_complete}")

    training_complete = True

print(f"Fine-tuning elapsed time: {elapsed_train_sec / 3600:.2f} hours")
print(f"Best validation Macro-F1: {best_val_macro_f1:.5f}")

## 13. Restore the best validation checkpoint and export DistilBERT + tokenizer

In [ ]:
if not BEST_STATE_PATH.exists():
    raise FileNotFoundError(
        "Best validation weights are missing. The training cell must finish at least one validation pass."
    )

best_payload = torch.load(BEST_STATE_PATH, map_location="cpu", weights_only=False)
model.load_state_dict(best_payload["model_state_dict"])
model.eval()

MODEL_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(MODEL_EXPORT_DIR, safe_serialization=True)
tokenizer.save_pretrained(MODEL_EXPORT_DIR)

training_history_df = pd.DataFrame(history)
training_history_df.to_csv(ARTIFACTS_DIR / "training_history.csv", index=False)

if not training_history_df.empty and "val_macro_f1" in training_history_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(training_history_df["global_step"], training_history_df["val_macro_f1"], marker="o")
    axes[0].set_title("Validation Macro-F1")
    axes[0].set_xlabel("Optimizer step")
    axes[0].set_ylabel("Macro-F1")
    axes[1].plot(training_history_df["global_step"], training_history_df["val_loss"], marker="o")
    axes[1].set_title("Validation Loss")
    axes[1].set_xlabel("Optimizer step")
    axes[1].set_ylabel("Loss")
    fig.tight_layout()
    fig.savefig(ARTIFACTS_DIR / "training_history.png", dpi=180)
    plt.show()
    plt.close(fig)

print("Restored best validation weights:")
print("Best validation Macro-F1:", best_payload["best_val_macro_f1"])
print("Selected optimizer step:", best_payload["global_step"])
print("Model and tokenizer saved to:", MODEL_EXPORT_DIR)

## 14. Final held-out test evaluation

This evaluates the selected best-validation checkpoint on the 40,000 held-out test reviews. The test set is not used for training, checkpoint selection, or tuning.

In [ ]:
test_loss, test_metrics, y_test, y_pred = evaluate_loader(model, test_loader)

report_dict = classification_report(
    y_test,
    y_pred,
    labels=[0, 1, 2],
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report_dict).transpose()
report_df.to_csv(ARTIFACTS_DIR / "classification_report.csv", index=True)

cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2])
plt.figure(figsize=(7, 6))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES
)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Model B — DistilBERT Confusion Matrix (Held-out Test)")
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "confusion_matrix.png", dpi=180)
plt.show()
plt.close()

test_predictions_df = pd.DataFrame({
    "record_index": test_idx.astype(np.int64),
    "true_label_id": y_test.astype(np.int64),
    "predicted_label_id": y_pred.astype(np.int64),
    "true_label": [ID2LABEL[int(x)] for x in y_test],
    "predicted_label": [ID2LABEL[int(x)] for x in y_pred],
})
test_predictions_df.to_csv(ARTIFACTS_DIR / "test_predictions.csv", index=False)

print("Held-out test metrics:")
for name, value in test_metrics.items():
    print(f"{name}: {value:.5f}")
print(f"Test loss: {test_loss:.5f}")
display(report_df)

## 15. Error analysis, measured GPU inference latency, and parameter count

In [ ]:
test_texts_path = MODEL_B_CACHE / "test_texts.jsonl"
test_texts = None
if test_texts_path.exists():
    with test_texts_path.open("r", encoding="utf-8") as f:
        test_texts = [json.loads(line)["text"] for line in f]
    assert len(test_texts) == len(test_idx), "Test texts are not aligned with the test indices."

error_positions = np.flatnonzero(y_test != y_pred)
error_rows = []
for pos in error_positions[:100]:
    row = {
        "record_index": int(test_idx[pos]),
        "true_label": ID2LABEL[int(y_test[pos])],
        "predicted_label": ID2LABEL[int(y_pred[pos])],
    }
    if test_texts is not None:
        row["text"] = test_texts[int(pos)]
    error_rows.append(row)

pd.DataFrame(error_rows).to_csv(ARTIFACTS_DIR / "error_analysis_top100.csv", index=False)
print(f"Saved {len(error_rows)} misclassified examples for error analysis.")
if test_texts is None:
    print("Note: original review text was unavailable, so the error file contains record indices and labels only.")

# Match Model A's latency protocol:
# - same first 16 / first 128 held-out test examples
# - pad lengths to multiples of 64 (minimum 64)
# - 20 warm-up calls and 200 synchronized measurements
# - batch sizes 1 and 32
latency_collator = DistilBertCollator(
    tokenizer.pad_token_id,
    max_length=MAX_SEQ_LEN,
    pad_multiple=64,
)
latency_dataset = YelpMemmapDataset(MODEL_B_CACHE, test_idx)

sample_batches_bs1 = [
    latency_collator([latency_dataset[i]])
    for i in range(16)
]
sample_batches_bs32 = [
    latency_collator([latency_dataset[i] for i in range(batch_idx * 32, (batch_idx + 1) * 32)])
    for batch_idx in range(4)
]

def benchmark_distilbert_latency(model, batches, warmup=20, steps=200):
    model.eval()
    staged = [
        (
            batch["input_ids"].to(device, non_blocking=True),
            batch["attention_mask"].to(device, non_blocking=True),
        )
        for batch in batches
    ]

    with torch.inference_mode():
        for i in range(warmup):
            input_ids, attention_mask = staged[i % len(staged)]
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                _ = model(input_ids=input_ids, attention_mask=attention_mask).logits

        torch.cuda.synchronize(device)
        latencies_ms = []
        total_samples = 0

        for i in range(steps):
            input_ids, attention_mask = staged[i % len(staged)]
            torch.cuda.synchronize(device)
            started = time.perf_counter()
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                _ = model(input_ids=input_ids, attention_mask=attention_mask).logits
            torch.cuda.synchronize(device)
            latencies_ms.append((time.perf_counter() - started) * 1000.0)
            total_samples += int(input_ids.size(0))

    values = np.asarray(latencies_ms, dtype=np.float64)
    mean_batch_ms = float(values.mean())
    total_seconds = float(values.sum()) / 1000.0
    mean_batch_size = total_samples / max(steps, 1)
    return {
        "mean_ms_per_batch": mean_batch_ms,
        "p50_ms_per_batch": float(np.percentile(values, 50)),
        "p95_ms_per_batch": float(np.percentile(values, 95)),
        "mean_ms_per_sample": mean_batch_ms / max(mean_batch_size, 1.0),
        "samples_per_sec": total_samples / total_seconds if total_seconds > 0 else 0.0,
        "warmup_runs": int(warmup),
        "measured_runs": int(steps),
    }

latency_stats = {
    "bs1": benchmark_distilbert_latency(model, sample_batches_bs1),
    "bs32": benchmark_distilbert_latency(model, sample_batches_bs32),
    "protocol": {
        "test_examples": "same first 128 held-out test examples used by Model A",
        "padding_multiple": 64,
        "minimum_padded_length": 64,
        "warmup_runs": 20,
        "measured_runs": 200,
        "cuda_synchronization": True,
        "mixed_precision": "bf16" if USE_BF16 else "fp16",
    },
}

total_parameters = int(sum(p.numel() for p in model.parameters()))
trainable_parameters = int(sum(p.numel() for p in model.parameters() if p.requires_grad))

print("GPU inference latency:", latency_stats)
print(f"Total parameters: {total_parameters:,}")
print(f"Trainable parameters: {trainable_parameters:,}")

## 16. Save the measured experiment report and export a Streamlit-ready inference module

In [ ]:
split_summary = {
    split_name: {
        "size": int(EXPECTED_SPLITS[split_name]["size"]),
        "class_counts": EXPECTED_SPLITS[split_name]["class_counts"],
        "sha256": EXPECTED_SPLITS[split_name]["sha256"],
    }
    for split_name in ("train", "val", "test")
}

metrics_payload = {
    "task": "3-class Yelp review sentiment analysis",
    "model": {
        "name": "DistilBERT",
        "checkpoint": MODEL_CHECKPOINT,
        "architecture": "pretrained DistilBERT encoder + 3-class classification head",
        "total_parameters": total_parameters,
        "trainable_parameters": trainable_parameters,
    },
    "dataset": {
        "name": "Yelp Dataset",
        "source_url": "https://www.kaggle.com/datasets/yelp-dataset/yelp-dataset",
        "file": "yelp_academic_dataset_review.json",
        "license_note": "Yelp Dataset Agreement; verify the current terms and use only for permitted academic/non-commercial purposes.",
        "raw_records": EXPECTED_COUNTS["raw"],
        "valid_unique_reviews": EXPECTED_COUNTS["deduped"],
        "label_mapping": {
            "negative": "1-2 stars",
            "neutral": "3 stars",
            "positive": "4-5 stars",
        },
    },
    "splits": split_summary,
    "preprocessing": {
        "shared_tokenizer": TOKENIZER_FOR_SHARED_CACHE,
        "distilbert_vocab_verified_identical": True,
        "max_sequence_length": MAX_SEQ_LEN,
        "truncation": f"head {HEAD_TOKENS} + tail {TAIL_TOKENS}",
        "deduplication": "lowercase + collapse whitespace; 64-bit BLAKE2b digest; same Model A builder",
    },
    "training": {
        "epochs": NUM_EPOCHS,
        "training_examples": int(len(train_dataset)),
        "per_device_batch_size": PER_DEVICE_BATCH_SIZE,
        "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
        "effective_batch_size": EFFECTIVE_BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "warmup_ratio": WARMUP_RATIO,
        "best_validation_macro_f1": float(best_payload["best_val_macro_f1"]),
        "fine_tuning_time_seconds": float(elapsed_train_sec),
        "checkpoint_resume_supported": True,
    },
    "hardware": {
        "gpu": GPU_NAME,
        "gpu_vram_gb": round(GPU_VRAM_GB, 2),
        "bf16_supported": GPU_BF16,
        "mixed_precision": "bf16" if USE_BF16 else "fp16",
        "pytorch": torch.__version__,
        "cuda": torch.version.cuda,
    },
    "held_out_test": {
        "loss": float(test_loss),
        **{k: float(v) for k, v in test_metrics.items()},
        "classification_report": report_dict,
        "confusion_matrix": cm.tolist(),
    },
    "inference_latency_gpu": latency_stats,
    "notes": [
        "The test set was not used for training, checkpoint selection, or hyperparameter tuning.",
        "Exact Model A split membership was verified with SHA-256 before training.",
        "The full training split was used; no 150K balanced subsampling was performed.",
        "All metric and timing values are measured by this notebook run.",
    ],
}

(ARTIFACTS_DIR / "metrics.json").write_text(json.dumps(metrics_payload, indent=2), encoding="utf-8")
(ARTIFACTS_DIR / "training_config.json").write_text(
    json.dumps({"training": TRAINING_CONFIG, "fingerprint": CONFIG_FINGERPRINT}, indent=2),
    encoding="utf-8",
)

inference_module = r"""
from pathlib import Path
import time
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

LABELS = {0: "negative", 1: "neutral", 2: "positive"}
MAX_LENGTH = 256
HEAD_TOKENS = 192
TAIL_TOKENS = 64

def load_model(model_dir=None, device=None):
    model_dir = Path(model_dir) if model_dir else Path(__file__).resolve().parent / "model"
    device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(device)
    model.eval()
    return model, tokenizer, device

def _head_tail_ids(tokenizer, text):
    ids = tokenizer(text, add_special_tokens=True, truncation=False)["input_ids"]
    if len(ids) > MAX_LENGTH:
        ids = ids[:HEAD_TOKENS] + ids[-TAIL_TOKENS:]
    return ids

@torch.inference_mode()
def predict_sentiment(text, model, tokenizer, device=None):
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Please provide a non-empty review text.")

    total_started = time.perf_counter()
    device = torch.device(device or next(model.parameters()).device)
    ids = _head_tail_ids(tokenizer, text)
    input_ids = torch.tensor([ids], dtype=torch.long, device=device)
    attention_mask = torch.ones_like(input_ids)

    if device.type == "cuda":
        torch.cuda.synchronize()
    model_started = time.perf_counter()

    if device.type == "cuda":
        amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    else:
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits

    if device.type == "cuda":
        torch.cuda.synchronize()
    model_latency_ms = (time.perf_counter() - model_started) * 1000.0

    probabilities = torch.softmax(logits.float(), dim=-1)[0].cpu().tolist()
    predicted_id = max(range(len(probabilities)), key=probabilities.__getitem__)
    end_to_end_latency_ms = (time.perf_counter() - total_started) * 1000.0

    return {
        "sentiment": LABELS[predicted_id],
        "confidence": float(probabilities[predicted_id]),
        "probabilities": {LABELS[i]: float(probabilities[i]) for i in range(3)},
        "latency_ms": float(end_to_end_latency_ms),
        "model_latency_ms": float(model_latency_ms),
    }
"""
(MODEL_B_ROOT / "pretrained_inference.py").write_text(inference_module.strip() + "\n", encoding="utf-8")

print("Metrics:", ARTIFACTS_DIR / "metrics.json")
print("Training config:", ARTIFACTS_DIR / "training_config.json")
print("Streamlit inference module:", MODEL_B_ROOT / "pretrained_inference.py")
print("Saved model:", MODEL_EXPORT_DIR)

## 17. Optional side-by-side comparison with Model A

If Model A's `artifacts_v2/metrics.json` is available in the same Google Drive, this cell builds a comparison table and chart using the measured held-out test metrics. It reports both Model A raw predictions and its validation-selected logit-adjusted predictions transparently. Model B is reported without post-hoc test tuning.

In [ ]:
MODEL_A_METRICS_PATH = DRIVE_ROOT / "yelp_model_a" / "artifacts_v2" / "metrics.json"

if MODEL_A_METRICS_PATH.exists():
    model_a_metrics = json.loads(MODEL_A_METRICS_PATH.read_text(encoding="utf-8"))

    a_raw = model_a_metrics["test_raw"]
    a_adjusted = model_a_metrics["test_logit_adjusted"]
    a_params = model_a_metrics["parameters"]["total"]
    a_train_seconds = model_a_metrics["training"]["elapsed_train_sec"]
    a_latency = model_a_metrics["gpu_latency_bf16"]["bs1"]["mean_ms_per_sample"]
    a_gpu_name = model_a_metrics["hardware"]["gpu_name"]

    b_test = metrics_payload["held_out_test"]
    b_params = metrics_payload["model"]["total_parameters"]
    b_train_seconds = metrics_payload["training"]["fine_tuning_time_seconds"]
    b_latency = metrics_payload["inference_latency_gpu"]["bs1"]["mean_ms_per_sample"]
    b_gpu_name = GPU_NAME

    def a_macro_precision_recall(metric_block):
        per_class = metric_block["per_class"]
        macro_precision = float(np.mean([v["precision"] for v in per_class.values()]))
        macro_recall = float(np.mean([v["recall"] for v in per_class.values()]))
        return macro_precision, macro_recall

    a_raw_precision, a_raw_recall = a_macro_precision_recall(a_raw)
    a_adj_precision, a_adj_recall = a_macro_precision_recall(a_adjusted)

    comparison_rows = [
        {
            "model": "Model A — Scratch (raw)",
            "accuracy": a_raw["accuracy"],
            "macro_precision": a_raw_precision,
            "macro_recall": a_raw_recall,
            "macro_f1": a_raw["macro_f1"],
            "parameters": a_params,
            "training_time_min": a_train_seconds / 60,
            "avg_inference_latency_ms": a_latency,
        },
        {
            "model": "Model A — Scratch (validation-adjusted)",
            "accuracy": a_adjusted["accuracy"],
            "macro_precision": a_adj_precision,
            "macro_recall": a_adj_recall,
            "macro_f1": a_adjusted["macro_f1"],
            "parameters": a_params,
            "training_time_min": a_train_seconds / 60,
            "avg_inference_latency_ms": a_latency,
        },
        {
            "model": "Model B — Pretrained DistilBERT",
            "accuracy": b_test["accuracy"],
            "macro_precision": b_test["macro_precision"],
            "macro_recall": b_test["macro_recall"],
            "macro_f1": b_test["macro_f1"],
            "parameters": b_params,
            "training_time_min": b_train_seconds / 60,
            "avg_inference_latency_ms": b_latency,
        },
    ]

    comparison_df = pd.DataFrame(comparison_rows)
    comparison_df.to_csv(ARTIFACTS_DIR / "model_comparison.csv", index=False)
    display(comparison_df)

    plot_df = comparison_df.set_index("model")[["accuracy", "macro_f1"]]
    ax = plot_df.plot(kind="bar", figsize=(11, 5), ylim=(0, 1), rot=12)
    ax.set_title("Model A vs Model B — Held-out Test Metrics")
    ax.set_ylabel("Score")
    ax.set_xlabel("")
    ax.legend(["Accuracy", "Macro-F1"])
    plt.tight_layout()
    plt.savefig(ARTIFACTS_DIR / "model_comparison.png", dpi=180)
    plt.show()
    plt.close()
    print("Comparison table and chart saved in:", ARTIFACTS_DIR)

    # Primary quality comparison is raw argmax vs raw argmax.
    # Model A's validation-adjusted score is shown separately as an additional result.
    delta_f1 = float(b_test["macro_f1"] - a_raw["macro_f1"])
    delta_accuracy = float(b_test["accuracy"] - a_raw["accuracy"])

    print("\n=== Measured comparison interpretation ===")
    print(
        f"Raw Macro-F1: Model B {b_test['macro_f1']:.4f} vs "
        f"Model A {a_raw['macro_f1']:.4f} (absolute difference {delta_f1:+.4f})."
    )
    print(
        f"Raw Accuracy: Model B {b_test['accuracy']:.4f} vs "
        f"Model A {a_raw['accuracy']:.4f} (absolute difference {delta_accuracy:+.4f})."
    )
    print(
        f"Model A validation-adjusted Macro-F1 on test: {a_adjusted['macro_f1']:.4f} "
        "(secondary result; its bias values were selected on validation data)."
    )
    if delta_f1 > 0:
        print("On this measured run, pretrained initialization improved raw Macro-F1 over Model A.")
    elif delta_f1 < 0:
        print("On this measured run, pretrained initialization did not improve raw Macro-F1 over Model A.")
    else:
        print("On this measured run, raw Macro-F1 was equal for both models.")

    if a_gpu_name != b_gpu_name:
        print(
            f"Latency caution: Model A ran on {a_gpu_name}, while Model B ran on {b_gpu_name}. "
            "Quality metrics remain comparable; hardware latency should not be directly ranked."
        )
    else:
        print(
            "Latency uses the same test examples, padding multiple, batch sizes, warm-up count, "
            "measurement count, and synchronized GPU timing protocol as Model A."
        )

    print(
        "Limitations: Model A used 3 epochs and Model B used "
        f"{NUM_EPOCHS} epoch(s); training budgets differ and must be disclosed. "
        "No performance advantage is assumed unless the measured metrics show it."
    )
else:
    print(
        "Model A metrics file not found at:", MODEL_A_METRICS_PATH,
        "\nThe Model B evaluation is complete, but the side-by-side comparison will be generated "
        "after Model A's artifacts_v2/metrics.json is shared in Drive."
    )

## 18. Smoke-test the exported inference module on held-out review examples

In [ ]:
import importlib
import sys

if str(MODEL_B_ROOT) not in sys.path:
    sys.path.insert(0, str(MODEL_B_ROOT))

import pretrained_inference
importlib.reload(pretrained_inference)

inference_model, inference_tokenizer, inference_device = pretrained_inference.load_model(
    model_dir=MODEL_EXPORT_DIR,
    device=device,
)

if test_texts is not None and len(test_texts) >= 3:
    demo_texts = test_texts[:3]
else:
    demo_texts = [
        "The staff were friendly and the food was excellent.",
        "The service was slow and the meal was disappointing.",
        "It was an average experience, nothing special.",
    ]

demo_rows = []
for review in demo_texts:
    prediction = pretrained_inference.predict_sentiment(
        review, inference_model, inference_tokenizer, inference_device
    )
    demo_rows.append({
        "review": review[:180],
        "predicted_sentiment": prediction["sentiment"],
        "confidence": prediction["confidence"],
        "end_to_end_latency_ms": prediction["latency_ms"],
        "model_latency_ms": prediction["model_latency_ms"],
    })

display(pd.DataFrame(demo_rows))
print("PASS: exported Model B inference module loaded and returned predictions.")

## 19. Final handoff checklist

- [ ] Train, validation, and test SHA-256 checks all printed `PASS`; all unique indices are covered exactly once.
- [ ] The complete 6,913,573-review training split was used (not the old 150K subset).
- [ ] Best checkpoint was selected using validation Macro-F1 only.
- [ ] The 40,000-review test set was evaluated only for final reporting.
- [ ] `yelp_model_b/model/` contains the exported DistilBERT model and tokenizer.
- [ ] `yelp_model_b/artifacts/` contains metrics, classification report, confusion matrix, predictions, error analysis, training history, and optional Model A comparison table/chart.
- [ ] Model A and Model B latency were measured with the same test examples and benchmark protocol.
- [ ] `yelp_model_b/pretrained_inference.py` is ready for Streamlit integration.
- [ ] `yelp_model_b/checkpoints/` contains the resumable training state.

Share this notebook and the resulting `yelp_model_b` Google Drive folder with the team. Never upload `kaggle.json` to GitHub. Keep the full dataset cache and large model weights on Drive rather than in the public repository.

For the final comparison, Model A and Model B must use the verified same test indices and label IDs. Report measured values only.